In [1]:
import sys
sys.path.append('..')

import tensorflow as tf
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.datasets import mnist 
import my_cnn
from scipy.ndimage import rotate
from sklearn.model_selection import train_test_split

# fixed seed so the results can be reproduced (my_cnn uses NumPy's global RNG)
SEED = 42
np.random.seed(SEED)

In [2]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# hold out 6,000 training images for validation (same split in every notebook);
# the test set is used only once, for the final evaluation
x_train, x_val, y_train, y_val = train_test_split(
    x_train, y_train, test_size=6000, stratify=y_train, random_state=SEED
)
print(f"train: {x_train.shape[0]:,}  validation: {x_val.shape[0]:,}  test: {x_test.shape[0]:,}")

x_train = np.expand_dims(x_train.astype(np.float32), axis=-1)
x_val = np.expand_dims(x_val.astype(np.float32), axis=-1)
x_test = np.expand_dims(x_test.astype(np.float32), axis=-1)
y_train = np.eye(10, dtype=np.float32)[y_train]
y_val = np.eye(10, dtype=np.float32)[y_val]
y_test = np.eye(10, dtype=np.float32)[y_test]

train: 54,000  validation: 6,000  test: 10,000


In [3]:
angles = [-12, 0, 12]
new_x = []
new_y = []

for img, label in zip(x_train, y_train):
    for angle in angles:
        img_rot = rotate(img, angle, axes=(0,1), reshape=False, mode='constant', cval=0.0)
        new_x.append(img_rot)
        new_y.append(label)

x_train_augmented = np.array(new_x)
y_train_augmented = np.array(new_y)


In [4]:
test_model = my_cnn.Model()
test_model.add(my_cnn.layers.Input((28, 28, 1)))
test_model.add(my_cnn.layers.Rescaling(1./255))
test_model.add(my_cnn.layers.Convolution2D(32, 3, activation_func='relu'))
test_model.add(my_cnn.layers.MaxPooling2D(2))
test_model.add(my_cnn.layers.Convolution2D(64, 3, activation_func='relu'))
test_model.add(my_cnn.layers.MaxPooling2D(2))
test_model.add(my_cnn.layers.Flatten())
test_model.add(my_cnn.layers.Dense(64, activation_func='relu'))
test_model.add(my_cnn.layers.Dense(10, activation_func='softmax'))

In [5]:
test_model.compile(optimizer='momentum', loss='crossentropy', learning_rate=0.02)
test_model.fit(x_train_augmented, y_train_augmented, epochs=2, batch_size=128)

Epoch 1/2 | Batch 1266/1266 | Loss: 0.1130 | Acc: 0.9645
Epoch 2/2 | Batch 1266/1266 | Loss: 0.0407 | Acc: 0.9870


In [6]:
val_loss, val_accuracy = test_model.evaluate(x_val, y_val)
print(f"Validation loss: {val_loss:.4f}")
print(f"Validation accuracy: {val_accuracy*100:.2f}")

Validation loss: 0.0463
Validation accuracy: 98.58


In [7]:
model_loss, model_accuracy = test_model.evaluate(x_test, y_test)

In [8]:
print(f"Loss: {model_loss}")
print(f"Accuracy: {model_accuracy*100:.2f}")

Loss: 0.03316180408000946
Accuracy: 98.94


In [9]:
predictions = test_model.predict(x_test)
decisions = np.argmax(predictions, axis=1)
y_test_classes = np.argmax(y_test, axis=1)

print(classification_report(y_test_classes, decisions))


              precision    recall  f1-score   support

           0       0.99      0.99      0.99       980
           1       0.99      1.00      0.99      1135
           2       1.00      0.99      0.99      1032
           3       0.98      1.00      0.99      1010
           4       0.99      1.00      0.99       982
           5       0.98      0.99      0.98       892
           6       0.99      0.99      0.99       958
           7       0.98      0.99      0.99      1028
           8       0.99      0.98      0.98       974
           9       0.99      0.97      0.98      1009

    accuracy                           0.99     10000
   macro avg       0.99      0.99      0.99     10000
weighted avg       0.99      0.99      0.99     10000



In [10]:
cm = confusion_matrix(y_test_classes, decisions)
df_cm = pd.DataFrame(cm)
df_cm.index.name = "Real"
df_cm.columns.name = "Predicted"
df_cm 

Predicted,0,1,2,3,4,5,6,7,8,9
Real,,,,,,,,,,
0,975,1,0,0,0,0,2,2,0,0
1,0,1134,0,1,0,0,0,0,0,0
2,3,1,1023,1,1,0,0,3,0,0
3,0,0,0,1009,0,0,0,0,1,0
4,0,1,0,0,978,0,0,0,0,3
5,1,0,0,9,0,879,2,0,1,0
6,1,3,0,0,1,5,948,0,0,0
7,0,3,3,0,0,0,0,1021,1,0
8,3,0,2,5,0,5,2,3,952,2


In [11]:
test_model.save('../models/custom_cnn.pkl')

Model saved to: ../models/custom_cnn.pkl
